<div dir="rtl">

# גישה (ג): מודל נלמד, מול "רואים חפץ" ומול הכללים

שאלת המחקר משווה שלוש גישות: (א) אובייקט בלבד, (ב) כללים שנכתבו ביד, (ג) **מודל שלומד** מרצפי שלד ואובייקטים. המחברת הזו בונה את (ג), **בלי תיוג ידני**:

1. **סרטוני אימון:** סרטוני UCF-Crime ש-**UCA** כבר מתאר עם זמנים, למשל "אדם מכוון אקדח, שנייה 12-18", ושאינם בין 107 סרטוני הבדיקה (`live/train_set.py`). יש כ-5 שעות וידאו עם כ-156 אירועי אקדח וכ-93 אירועי אש, ועוד שעה של סרטונים רגילים.
2. **perception:** אותו שלד ואותם גלאים שהמערכת מריצה היום (`security.pt` לאש, `security_v3.pt` לאקדח וסכין).
3. **אימון:** כל 2 שניות של וידאו הופכות למספרים: איפה הידיים ביחס לכתפיים, המרחק בין היד לחפץ, הביטחון של הגלאי ומהירות התנועה. מודל gradient boosting לומד מתי זה אירוע (`live/learned.py`).
4. **בדיקה:** על אותם 107 סרטונים ובאותם מדדים כמו (א) ו-(ב), עם רווחי בר-סמך והשוואות מזווגות באותה כמות של התרעות שווא.

**לפני שמתחילים:** Runtime → Change runtime type → **T4 GPU**. ואז Runtime → **Run all**.

**אין GPU פנוי?** אפשר להתחיל בלעדיו. תאים 1-4 (ההורדה, החלק הארוך) רצים גם בלי GPU, ותא 5 יעצור עם הודעה. כש-GPU מתפנה, מחליפים ל-T4 ומריצים Run all שוב, ומה שכבר ירד מדולג.

**זמן:** הורדה של כ-5GB, כשעה עד שעתיים (השרת של UCF איטי). perception כשעה. אימון ובדיקה כמה דקות. הכול נשמר ב-Drive, ואם Colab מתנתק מריצים Run all שוב והמחברת ממשיכה מאיפה שעצרה.

</div>

In [ ]:
# 1. GPU, Drive, packages
%cd /content
import os, torch
GPU = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if GPU else
      'אין. זה בסדר לתאים 3-4 (ההורדה, החלק הארוך). תא 5 צריך GPU: כשיתפנה, Runtime → Change runtime type → T4 GPU ו-Run all')
from google.colab import drive
drive.mount('/content/drive')
LE = '/content/drive/MyDrive/live_eval'
os.makedirs(LE, exist_ok=True)
!pip -q install ultralytics lap remotezip truststore scikit-learn joblib

In [ ]:
# 2. Code
BRANCH = 'live-action-alerts'
%cd /content
!rm -rf /content/final_project
!git clone -q -b {BRANCH} https://github.com/Talyaavraham/final_project.git /content/final_project
%cd /content/final_project/backend
!ls security.pt security_v3.pt live/learned.py live/train_set.py

In [ ]:
# 3. The training videos (none of them is a test video) + where they are in the UCF-Crime archive
!python live/train_set.py --scenarios weapon,fire --max-minutes 300 --negative-minutes 60 --out eval/ground_truth_train.json
import json, truststore
truststore.inject_into_ssl()
from remotezip import RemoteZip
gt = json.load(open('eval/ground_truth_train.json'))['videos']
want = {os.path.splitext(v)[0] for v in gt}
with RemoteZip('https://www.crcv.ucf.edu/data1/chenchen/UCF_Crimes.zip', timeout=(30, 120)) as z:
    infos = {os.path.splitext(os.path.basename(i.filename))[0]: i for i in z.infolist() if i.filename.endswith('.mp4')}
sel = {n: {'path': infos[n].filename, 'bytes': infos[n].file_size} for n in sorted(want) if n in infos}
json.dump(sel, open('eval/data/ucf_selection_train.json', 'w'), indent=1)
print(f"{len(sel)} of {len(want)} in the archive, {sum(v['bytes'] for v in sel.values()) / 1e9:.1f} GB to download")

In [ ]:
# 4. Download to Drive (resumable: run again after a disconnect)
VID = f'{LE}/ucf_videos_train'
!python eval/fetch_ucf.py --sel eval/data/ucf_selection_train.json --out {VID} --workers 4
!ls {VID} | wc -l

In [ ]:
# 5. Perception of the training videos with today's detectors (resumable, ~1 h on a T4)
import torch
if not torch.cuda.is_available():
    raise SystemExit('תא 5 צריך GPU. ההורדה נשמרה ב-Drive. כש-T4 יתפנה: Runtime → Change runtime type → T4 GPU, ואז Run all (מה שכבר ירד מדולג).')
os.environ['LIVE_EXTRA'] = 'security.pt=fire+smoke,security_v3.pt=gun+knife'
os.environ['LIVE_DEVICE'] = '0'
TRAIN_P = f'{LE}/perception_fps3_combo_train'
!python live/perceive.py --gt eval/ground_truth_train.json --videos {VID} --out {TRAIN_P} --fps 3

In [ ]:
# 6. The 107 TEST videos with the same detectors: merged from the two runs already on Drive (seconds)
#    perception_fps3 = security.pt (fire / smoke from here), perception_fps3_sec3 = security_v3.pt (the rest)
TEST_P = f'{LE}/perception_fps3_combo'
!python live/merge_perception.py --base {LE}/perception_fps3_sec3 --from {LE}/perception_fps3 --take fire,smoke --out {TEST_P}

In [ ]:
# 7. Approaches (a) and (b) on the same test cache: every method of scenarios.json at several detector thresholds
for c in (0.25, 0.35, 0.45, 0.55, 0.65):
    !python live/eval_alerts.py --gt eval/ground_truth_uca.json --perception {TEST_P} --scenarios weapon,fire --conf {c} --name rules_combo_conf{c} | grep -E "weapon|fire"

In [ ]:
# 8. Approach (c): train on the training videos, test on the 107
for sc in ('weapon', 'fire'):
    print(f'\n#################### {sc}')
    !python live/learned.py train --gt eval/ground_truth_train.json --perception {TRAIN_P} --scenario {sc} --out live/models/learned_{sc}.joblib
    !python live/learned.py eval --gt eval/ground_truth_uca.json --perception {TEST_P} --model live/models/learned_{sc}.joblib --name learned_{sc}

In [ ]:
# 9. (c) vs (a) and (b) at a similar number of false alarms: paired bootstrap on the same videos
import sys, glob; sys.path.insert(0, 'live')
import compare
rules = {}
for p in glob.glob('live/results/rules_combo_conf*.json'):
    c = p.split('conf')[-1][:-5]
    for sc, ms in json.load(open(p))['scenarios'].items():
        for m, r in ms.items():
            rules[(sc, f'{m} @{c}')] = r
for sc in ('weapon', 'fire'):
    learned = json.load(open(f'live/results/learned_{sc}.json'))['scenarios'][sc]
    print(f'\n==================== {sc}')
    for lname, L in learned.items():
        if not L['false_per_hour'] or L['caught'] == 0:
            continue
        # the rule methods whose false alarms per hour are closest to the learned model's
        cands = sorted(((abs(r['false_per_hour'] - L['false_per_hour']), k, r) for k, r in rules.items()
                        if k[0] == sc and r['false_per_hour'] is not None), key=lambda x: x[0])[:2]
        for _, (s, m), r in cands:
            out = compare.paired(L['per_video'], r['per_video'], n=3000)
            d, lo, hi = out['d_recall']
            verdict = ('learned catches more' if lo > 0 else 'rule catches more' if hi < 0 else 'cannot tell apart')
            print(f"{lname:16s} {L['recall'] * 100:5.1f}% @{L['false_per_hour']:5.1f}/h  vs  {m:24s} "
                  f"{r['recall'] * 100:5.1f}% @{r['false_per_hour']:5.1f}/h  ->  {d * 100:+5.1f} pts "
                  f"[{lo * 100:+.1f}, {hi * 100:+.1f}]  {verdict}")

In [ ]:
# 10. Keep the results and the models on Drive, and download the tables
!mkdir -p {LE}/learned && cp live/results/learned_* live/results/rules_combo_* {LE}/learned/ && cp live/models/*.joblib {LE}/learned/
from google.colab import files
for p in sorted(glob.glob('live/results/learned_*.txt') + glob.glob('live/results/rules_combo_*.txt')):
    files.download(p)